# Spider DB-ID Predictor — Colab GPU Training

This notebook fine-tunes the **question → database-schema bi-encoder** used by the NLQ-to-SQL pipeline.

### What it does
- Uses a Sentence-Transformer bi-encoder (`BAAI/bge-base-en-v1.5` by default).
- Trains with `MultipleNegativesRankingLoss`.
- Uses **duplicate-free batches** so the in-batch negatives do not contain duplicate positive schemas.
- Uses Colab GPU automatically, with FP16/BF16 mixed precision and TF32 where supported.
- Evaluates **Top-1 / Top-3 / Top-5 / Top-10 / MRR** on Spider dev.
- Saves resumable checkpoints to Google Drive.
- Saves the final model directly in the path expected by your existing pipeline: `models/db_id_predictor`.

> Important: improving DB retrieval is necessary but does not by itself guarantee >94% end-to-end Spider execution accuracy. The notebook targets the DB-selection bottleneck while keeping the existing Top-10 → Schema Agent architecture.

In [ ]:
# 1. Check the Colab GPU
import os, sys, subprocess, json, platform, torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("WARNING: No GPU detected. In Colab use Runtime → Change runtime type → GPU.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA L4
VRAM: 22.0 GB
BF16 supported: True


In [ ]:
# 2. Install the training stack
# Restart the runtime only if Colab asks you to after upgrading packages.

!pip -q install -U "sentence-transformers>=5.0.0,<6.0.0" "datasets>=3.0.0,<5.0.0" accelerate

import sentence_transformers, datasets, accelerate
print("sentence-transformers:", sentence_transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

sentence-transformers: 5.7.0
datasets: 4.8.5
accelerate: 1.15.0


## 3. Put Spider data on Google Drive

Expected files:

```text
/content/drive/MyDrive/spider/
├── train_spider.json
├── dev.json                 # optional but recommended
└── tables.json
```

The notebook only needs these three files for DB-ID training/evaluation. Your end-to-end pipeline still needs the Spider SQLite database directory separately.

If your files are elsewhere, change the paths in the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 3. Mount Google Drive and configure paths
# from google.colab import drive
# drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/nl2sql_pipeline"
SPIDER_ROOT = "/content/drive/MyDrive/spider_data/spider_data"

TRAIN_PATH = f"{SPIDER_ROOT}/train_spider.json"
DEV_PATH = f"{SPIDER_ROOT}/dev.json"
TABLES_PATH = f"{SPIDER_ROOT}/tables.json"

MODEL_OUTPUT = f"{PROJECT_ROOT}/models/db_id_predictor"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/models/db_id_predictor_checkpoints"

BASE_MODEL = "BAAI/bge-base-en-v1.5"

for p in [TRAIN_PATH, TABLES_PATH]:
    assert os.path.exists(p), f"Missing required file: {p}"

print("Train:", TRAIN_PATH)
print("Dev exists:", os.path.exists(DEV_PATH))
print("Tables:", TABLES_PATH)
print("Final model:", MODEL_OUTPUT)
print("Checkpoints:", CHECKPOINT_DIR)

Train: /content/drive/MyDrive/spider_data/spider_data/train_spider.json
Dev exists: True
Tables: /content/drive/MyDrive/spider_data/spider_data/tables.json
Final model: /content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor
Checkpoints: /content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor_checkpoints


In [ ]:
# 4. Load and inspect Spider
import json, os, statistics
from collections import Counter

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_rows = json.load(f)
with open(TABLES_PATH, "r", encoding="utf-8") as f:
    tables_rows = json.load(f)

dev_rows = []
if os.path.exists(DEV_PATH):
    with open(DEV_PATH, "r", encoding="utf-8") as f:
        dev_rows = json.load(f)

print("Train examples:", len(train_rows))
print("Dev examples:", len(dev_rows))
print("Databases:", len(tables_rows))
print("Train DB coverage:", len(set(x["db_id"] for x in train_rows)))
print("Dev DB coverage:", len(set(x["db_id"] for x in dev_rows)))

print("\nMost frequent training DBs:")
for db, n in Counter(x["db_id"] for x in train_rows).most_common(10):
    print(f"{db:35} {n}")

Train examples: 7000
Dev examples: 1034
Databases: 166
Train DB coverage: 140
Dev DB coverage: 20

Most frequent training DBs:
college_2                           170
college_1                           164
hr_1                                124
store_1                             112
soccer_2                            106
bike_1                              104
music_1                             100
hospital_1                          100
music_2                             100
dorm_1                              100


In [ ]:
# 5. Build the exact schema representation used by the runtime predictor
from typing import Any

def database_schema_text(item: dict[str, Any]) -> str:
    tables = item["table_names_original"]
    columns = item["column_names_original"]
    column_types = item["column_types"]
    primary_keys = set(item["primary_keys"])
    foreign_keys = item["foreign_keys"]

    lines = []
    for table_idx, table_name in enumerate(tables):
        lines.append(f"TABLE: {table_name}")
        for col_idx, (table_id, col_name) in enumerate(columns):
            if table_id != table_idx:
                continue
            line = f"  COLUMN: {col_name} TYPE: {column_types[col_idx]}"
            if col_idx in primary_keys:
                line += " PRIMARY_KEY"
            lines.append(line)

    if foreign_keys:
        lines.append("FOREIGN_KEYS:")
        for source_col, target_col in foreign_keys:
            st, sc = columns[source_col]
            tt, tc = columns[target_col]
            lines.append(
                f"  {tables[st]}.{sc} -> {tables[tt]}.{tc}"
            )
    return "\n".join(lines)

CATALOG = {x["db_id"]: database_schema_text(x) for x in tables_rows}

def make_pairs(rows):
    return [
        {
            "question": x["question"],
            "schema": CATALOG[x["db_id"]],
            "db_id": x["db_id"],
        }
        for x in rows
        if x["db_id"] in CATALOG
    ]

train_pairs = make_pairs(train_rows)
dev_pairs = make_pairs(dev_rows)

print("Training pairs:", len(train_pairs))
print("Dev pairs:", len(dev_pairs))
print("\nExample:")
print("QUESTION:", train_pairs[0]["question"])
print("SCHEMA:\n", train_pairs[0]["schema"][:1500])

Training pairs: 7000
Dev pairs: 1034

Example:
QUESTION: How many heads of the departments are older than 56 ?
SCHEMA:
 TABLE: department
  COLUMN: Department_ID TYPE: number PRIMARY_KEY
  COLUMN: Name TYPE: text
  COLUMN: Creation TYPE: text
  COLUMN: Ranking TYPE: number
  COLUMN: Budget_in_Billions TYPE: number
  COLUMN: Num_Employees TYPE: number
TABLE: head
  COLUMN: head_ID TYPE: number PRIMARY_KEY
  COLUMN: name TYPE: text
  COLUMN: born_state TYPE: text
  COLUMN: age TYPE: number
TABLE: management
  COLUMN: department_ID TYPE: number PRIMARY_KEY
  COLUMN: head_ID TYPE: number
  COLUMN: temporary_acting TYPE: text
FOREIGN_KEYS:
  management.head_ID -> head.head_ID
  management.department_ID -> department.Department_ID


In [ ]:
# 6. Choose a GPU-friendly batch size automatically
gpu_mem_gb = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if torch.cuda.is_available() else 0
)

if gpu_mem_gb >= 35:
    BATCH_SIZE = 64
elif gpu_mem_gb >= 20:
    BATCH_SIZE = 48
else:
    BATCH_SIZE = 32

GRAD_ACCUM = 1
EPOCHS = 4
LEARNING_RATE = 2e-5
MAX_SEQ_LENGTH = 512
SAVE_STEPS = 500

USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)

os.makedirs(MODEL_OUTPUT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print({
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "bf16": USE_BF16,
    "fp16": USE_FP16,
    "checkpoint_every_steps": SAVE_STEPS,
})

{'gpu': 'NVIDIA L4', 'batch_size': 48, 'gradient_accumulation': 1, 'effective_batch_size': 48, 'epochs': 4, 'learning_rate': 2e-05, 'max_seq_length': 512, 'bf16': True, 'fp16': False, 'checkpoint_every_steps': 500}


## 7. Train the bi-encoder

The important change versus the old training script is **duplicate-free in-batch negatives**.

With `MultipleNegativesRankingLoss`, every other schema in a batch acts as a negative. Spider contains many questions belonging to the same database, so ordinary random batching can accidentally put the same DB schema in the negative set. `BatchSamplers.NO_DUPLICATES` avoids that failure mode.

The trainer also writes checkpoints to Drive, so a Colab disconnect does not force you to start from zero.

In [ ]:
# 7. Create the model, dataset, loss and trainer
import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerTrainer,
    losses,
)

train_ds = Dataset.from_dict({
    "sentence_0": [x["question"] for x in train_pairs],
    "sentence_1": [x["schema"] for x in train_pairs],
})

model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LENGTH

# Helps Ampere/Ada GPUs use Tensor Cores for matrix operations.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.10,
    fp16=USE_FP16,
    bf16=USE_BF16,
    batch_sampler="no_duplicates",
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    dataloader_num_workers=2,
    tf32=torch.cuda.is_available(),
    seed=42,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    loss=loss,
)

print("Ready to train.")
print("Checkpoints will be written under:", CHECKPOINT_DIR)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Ready to train.
Checkpoints will be written under: /content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor_checkpoints


In [ ]:
# 8. Find the newest checkpoint, if any
import glob, os, re

def newest_checkpoint(folder):
    paths = glob.glob(os.path.join(folder, "checkpoint-*"))
    if not paths:
        return None
    def step(p):
        m = re.search(r"checkpoint-(\d+)$", p)
        return int(m.group(1)) if m else -1
    return max(paths, key=step)

RESUME_CHECKPOINT = newest_checkpoint(CHECKPOINT_DIR)
print("Resume checkpoint:", RESUME_CHECKPOINT or "None — training from base model")

Resume checkpoint: None — training from base model


In [ ]:
# 9. Train / resume
if RESUME_CHECKPOINT:
    print("Resuming:", RESUME_CHECKPOINT)
    trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
else:
    trainer.train()

print("Training finished.")

Step,Training Loss
25,1.783714
50,0.902423
75,0.690565
100,0.532801
125,0.501591
150,0.429255
175,0.315667
200,0.346761
225,0.331083
250,0.329177


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished.


In [ ]:
# 10. Save the final model at the exact path used by the pipeline
model.save(MODEL_OUTPUT)

print("FINAL MODEL SAVED:", MODEL_OUTPUT)
print("Contents:")
for name in sorted(os.listdir(MODEL_OUTPUT))[:30]:
    print(" ", name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FINAL MODEL SAVED: /content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor
Contents:
  1_Pooling
  2_Normalize
  README.md
  config.json
  config_sentence_transformers.json
  model.safetensors
  modules.json
  sentence_bert_config.json
  tokenizer.json
  tokenizer_config.json


## 11. Evaluate DB retrieval on Spider dev

These are the metrics that matter for the DB-ID component. In the current architecture, **Top-10 recall is the ceiling for the Schema Agent**, because the correct DB must enter its candidate list first.

In [ ]:
# 11. Retrieval evaluation
import numpy as np

@torch.inference_mode()
def evaluate_retrieval(model, pairs, catalog, batch_size=64):
    db_ids = list(catalog.keys())
    schemas = [catalog[x] for x in db_ids]

    q_emb = model.encode(
        [x["question"] for x in pairs],
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    s_emb = model.encode(
        schemas,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    score_matrix = q_emb @ s_emb.T
    db_to_idx = {db: i for i, db in enumerate(db_ids)}

    out = {}
    for k in [1, 3, 5, 10]:
        hits = 0
        for i, row in enumerate(pairs):
            topk = np.argsort(-score_matrix[i])[:k]
            hits += int(db_to_idx[row["db_id"]] in topk)
        out[f"top{k}"] = hits / len(pairs)

    reciprocal_ranks = []
    for i, row in enumerate(pairs):
        order = np.argsort(-score_matrix[i])
        rank = int(np.where(order == db_to_idx[row["db_id"]])[0][0]) + 1
        reciprocal_ranks.append(1.0 / rank)
    out["mrr"] = float(np.mean(reciprocal_ranks))
    return out

if dev_pairs:
    metrics = evaluate_retrieval(model, dev_pairs, CATALOG, batch_size=BATCH_SIZE)
    print("\nSpider DB retrieval metrics")
    print("=" * 40)
    for k, v in metrics.items():
        print(f"{k:>6}: {v:.4%}")
else:
    print("dev.json not found; add it to evaluate against Spider dev.")

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Spider DB retrieval metrics
  top1: 80.1741%
  top3: 91.6828%
  top5: 95.1644%
 top10: 98.5493%
   mrr: 86.8818%


In [ ]:
# 12. Inspect a few retrieval mistakes
def show_errors(model, pairs, catalog, n=10):
    db_ids = list(catalog.keys())
    schemas = [catalog[x] for x in db_ids]
    q_emb = model.encode(
        [x["question"] for x in pairs],
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    s_emb = model.encode(
        schemas,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    scores = q_emb @ s_emb.T
    db_to_idx = {db: i for i, db in enumerate(db_ids)}

    errors = []
    for i, row in enumerate(pairs):
        order = np.argsort(-scores[i])
        if db_to_idx[row["db_id"]] != order[0]:
            errors.append((row, [(db_ids[j], float(scores[i, j])) for j in order[:5]]))

    for row, candidates in errors[:n]:
        print("\nQUESTION:", row["question"])
        print("GOLD:", row["db_id"])
        print("TOP-5:")
        for db, score in candidates:
            print(f"  {db:35} {score:.5f}")

if dev_pairs:
    show_errors(model, dev_pairs, CATALOG, n=10)


QUESTION: How many singers do we have?
GOLD: concert_singer
TOP-5:
  singer                              0.63076
  concert_singer                      0.54799
  music_2                             0.51524
  music_1                             0.32485
  music_4                             0.30559

QUESTION: What is the total number of singers?
GOLD: concert_singer
TOP-5:
  singer                              0.65542
  concert_singer                      0.59731
  music_2                             0.47884
  music_1                             0.40484
  music_4                             0.32253

QUESTION: Show name, country, age for all singers ordered by age from the oldest to the youngest.
GOLD: concert_singer
TOP-5:
  singer                              0.65740
  concert_singer                      0.61557
  music_2                             0.43916
  music_1                             0.43293
  music_4                             0.39994

QUESTION: What are the names, countrie

## 13. Connect the model to the existing pipeline

Your updated runtime `src/db_prediction/model.py` supports:

```bash
DB_MODEL_PATH=/content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor
SPIDER_TABLES_PATH=/content/drive/MyDrive/spider/tables.json
```

When the project is copied from Drive into the Colab runtime, the predictor automatically uses CUDA if available.

For local/production inference, keep the same directory structure:

```text
models/
└── db_id_predictor/
    ├── config.json
    ├── modules.json
    ├── model.safetensors
    └── ...
```

Do **not** point `DB_MODEL_PATH` at a training checkpoint unless you deliberately want to test that checkpoint. Use the final `db_id_predictor` directory saved in the previous cell.

In [ ]:
# 13. Optional: quick runtime smoke test
import sys, os

# If this notebook is being run from a Drive copy of your project:
PROJECT_SRC = f"{PROJECT_ROOT}"
if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

os.environ["DB_MODEL_PATH"] = MODEL_OUTPUT
os.environ["SPIDER_TABLES_PATH"] = TABLES_PATH

# Import after setting environment variables.
from src.db_prediction.model import DBIDPredictor

predictor = DBIDPredictor(
    model_path=MODEL_OUTPUT,
    tables_path=TABLES_PATH,
)

test_question = dev_pairs[0]["question"] if dev_pairs else train_pairs[0]["question"]
print("QUESTION:", test_question)
for item in predictor.predict(test_question, top_k=10):
    print(f"{item['rank']:2}. {item['db_id']:35} {item['bi_score']:.5f}")

### Recommended training workflow

1. Run the notebook with a GPU.
2. Let it save checkpoints to Drive.
3. Evaluate Top-1/Top-5/Top-10 on Spider dev.
4. Keep the final model at `models/db_id_predictor`.
5. Use that directory as `DB_MODEL_PATH` in the existing pipeline.
6. For end-to-end Spider execution accuracy, separately evaluate the full NLQ → Schema Agent → Query Agent → execution pipeline.

If a Colab session disconnects, rerun through the configuration cells and the notebook will detect the newest `checkpoint-*` directory and resume training.

In [ ]:
import shutil
from google.colab import files

# Define paths
model_folder = '/content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor'
zip_output_path = '/content/db_id_predictor'

print("Compressing the model folder... This might take a minute.")
# Create db_id_predictor.zip
shutil.make_archive(zip_output_path, 'zip', model_folder)

print("Compression complete. Starting download...")
# Trigger browser download
files.download(f"{zip_output_path}.zip")

Compressing the model folder... This might take a minute.
Compression complete. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>